# World's Top Incomes (WTID) – Solution

**Short name (GitHub):** `TopInc_Py`  
**Lab source:** Packt *Practical Data Science Cookbook* 2e, Chapter 3.  
**Data:** `data/income_dist.csv` (2,180 × 354).  

This notebook is the worked companion to `TopInc_Py_Practice_Skeleton.ipynb`. Alternate pandas / R-shaped code sits next to the Cookbook generator style.


## Inline cheat-sheet (keep this cell visible)

See also **`TopInc_Py_Cheatsheet.docx`**. Short name: **`TopInc_Py`**.

| Task | Pattern |
|------|---------|
| Stream CSV | `csv.DictReader` + `yield row` (do not load 354 cols if you only need 6) |
| Filter country | `filter(lambda r: r["Country"]==name, reader)` or `df.query("Country == @name")` |
| Time series | `(int(row["Year"]), float(row[col]))` and skip blank strings |
| Mean-normalize | `vals / vals.mean()` so 1.0 = historical average for that fractile |
| Capital-gains lift | `share_incl_cg - share_excl_cg` (percentage points) |
| R column names | `read.csv(..., check.names=FALSE)` keeps `"Top 1% income share"` |
| Units | US average incomes are **real 2008 USD**; shares are **percent of total income** |
| Caveat | Fractiles excluding CG are **not** the same people as fractiles including CG |


![pipeline](topinc_py_flowchart.png)

## 0. Packages

In [ ]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data/income_dist.csv")
NOTES = Path("data/notes.csv")
plt.rcParams.update({"figure.figsize": (10, 4.4), "axes.grid": True, "grid.alpha": 0.25})
print("ready", DATA.exists())


## 1. Import with the standard library

In [ ]:
with open(DATA, newline="") as f:
    reader = csv.DictReader(f)
    data = list(reader)
    n_fields = len(reader.fieldnames)
print(len(data), n_fields, list(data[0].keys())[:4])
assert len(data) == 2180
assert n_fields == 354


In [ ]:
def dataset(path, filter_field=None, filter_value=None):
    """Yield CSV rows, optionally filtered on one field."""
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        if filter_field is None:
            yield from reader
        else:
            yield from (row for row in reader if row[filter_field] == filter_value)

row0 = next(dataset(DATA))
print(row0["Country"], row0["Year"])


In [ ]:
countries = {row["Country"] for row in dataset(DATA)}
years = {int(row["Year"]) for row in dataset(DATA)}
print("n countries", len(countries))
print(sorted(countries))
print("years", min(years), max(years))


## 2. United States average income

In [ ]:
def _to_float(x):
    if x is None or str(x).strip() == "":
        return None
    try:
        return float(x)
    except ValueError:
        return None

us_avg = []
for row in dataset(DATA, "Country", "United States"):
    val = _to_float(row["Average income per tax unit"])
    if val is not None:
        us_avg.append((int(row["Year"]), val))
print("US points", len(us_avg), "span", us_avg[0], us_avg[-1])
print("1913 / 1980 / 2007 / 2008",
      [v for y, v in us_avg if y in (1913, 1980, 2007, 2008)])


In [ ]:
years_u, vals_u = zip(*us_avg)
fig, ax = plt.subplots()
ax.bar(years_u, vals_u, color="#1f4e79", width=0.9)
ax.set_xticks(list(years_u)[::4])
ax.set_xticklabels(list(years_u)[::4], rotation=45)
ax.set_ylabel("Income in 2008 USD")
ax.set_title("U.S. Average Income per Tax Unit, 1913–2008")
fig.tight_layout()
fig.savefig("topinc_us_avg_income.png", dpi=140)
plt.show()


**Reading.** Real average income per U.S. tax unit rises from about **$14.6k** (1913) to **$40.7k** (1980) and **$54.1k** (2007), then drops to **$51.3k** in the 2008 crisis year. The bar chart is the Cookbook sanity plot — it does *not* yet say anything about the *distribution*.


## 3. Helpers

In [ ]:
def timeseries(rows, column):
    for row in rows:
        val = _to_float(row.get(column, ""))
        if val is not None:
            yield int(row["Year"]), val

def linechart(series, **kwargs):
    fig, ax = plt.subplots()
    labels = kwargs.get("labels") or []
    for i, line in enumerate(series):
        line = list(line)
        if not line:
            continue
        xs = [v[0] for v in line]
        ys = [v[1] for v in line]
        ax.plot(xs, ys, lw=2, label=labels[i] if i < len(labels) else None)
    if kwargs.get("ylabel"):
        ax.set_ylabel(kwargs["ylabel"])
    if kwargs.get("title"):
        ax.set_title(kwargs["title"])
    if labels:
        ax.legend(frameon=False, fontsize=8, ncol=min(len(labels), 5))
    fig.tight_layout()
    return fig


## 4. U.S. top shares

In [ ]:
SHARE_COLS = (
    "Top 10% income share",
    "Top 5% income share",
    "Top 1% income share",
    "Top 0.5% income share",
    "Top 0.1% income share",
)
us_rows = list(dataset(DATA, "Country", "United States"))
fig = linechart(
    [timeseries(us_rows, c) for c in SHARE_COLS],
    labels=[c.replace(" income share", "") for c in SHARE_COLS],
    ylabel="Percentage of total income",
    title="U.S. Top Income Shares, 1913–2008",
)
fig.savefig("topinc_us_shares.png", dpi=140)
plt.show()


**Reading.** All five fractiles fall from the late-1920s / WWII window into a postwar plateau, then rise together after ~1980. U.S. Top 1% share: **17.96% (1913) → 8.18% (1980) → 18.29% (2007) → 17.67% (2008)**. Top 10% (available from 1917): **40.29% → 32.87% (1980) → 45.51% (2007)**. Nested fractiles must move together; the interesting fact is *how much* the upper tail recovered.


In [ ]:
def normalize(ts):
    data = list(ts)
    arr = np.array([v[1] for v in data], dtype="f8")
    arr = arr / arr.mean()
    return list(zip((v[0] for v in data), arr))

fig = linechart(
    [normalize(timeseries(us_rows, c)) for c in SHARE_COLS],
    labels=[c.replace(" income share", "") for c in SHARE_COLS],
    ylabel="Share / long-run mean",
    title="Mean-Normalized U.S. Top Income Shares",
)
fig.savefig("topinc_us_shares_norm.png", dpi=140)
plt.show()


**Reading.** Dividing by each series' own mean puts them on a common scale. The top 0.1% path has the largest amplitude — the Cookbook point: *the richer the group, the larger the percentage-wise swings*.


In [ ]:
def delta(first, second):
    a = {y: v for y, v in first}
    b = {y: v for y, v in second}
    years = sorted(set(a) & set(b))
    return [(y, a[y] - b[y]) for y in years]

CG_PAIRS = (
    ("Top 10% income share-including capital gains", "Top 10% income share"),
    ("Top 5% income share-including capital gains", "Top 5% income share"),
    ("Top 1% income share-including capital gains", "Top 1% income share"),
    ("Top 0.5% income share-including capital gains", "Top 0.5% income share"),
    ("Top 0.1% income share-including capital gains", "Top 0.1% income share"),
)
fig = linechart(
    [delta(timeseries(us_rows, a), timeseries(us_rows, b)) for a, b in CG_PAIRS],
    labels=[b.replace(" income share", "") for a, b in CG_PAIRS],
    ylabel="Percentage-point lift",
    title="U.S. Capital-Gains Lift of Top Shares",
)
fig.savefig("topinc_us_cg_lift.png", dpi=140)
plt.show()


**Reading.** The lift spikes in boom years (late 1920s, 1980s–90s, 1999–2000, mid-2000s) and collapses in busts. That is capital-gains *realization*, not a smooth labor-income process. Do not treat "including CG" and "excluding CG" fractiles as the same people — WTID re-ranks when CG enter the income definition.


## 5. Further U.S. analysis

In [ ]:
AVG_COLS = (
    "Top 10% average income",
    "Top 5% average income",
    "Top 1% average income",
    "Top 0.5% average income",
    "Top 0.1% average income",
)
fig = linechart(
    [timeseries(us_rows, c) for c in AVG_COLS],
    labels=[c.replace(" average income", "") for c in AVG_COLS],
    ylabel="2008 USD",
    title="U.S. Average Income within Top Groups",
)
fig.savefig("topinc_us_avg_groups.png", dpi=140)
plt.show()


**Reading.** Levels stay compressed until the 1980s, then the top 0.1% average pulls away into the millions of 2008 dollars. Shares *and* averages are needed: a rising share on a rising mean is a larger dollar gap than either series alone shows.


In [ ]:
COMP_COLS = {
    "Salary": "Top 10% income composition-Wages, salaries and pensions",
    "Dividends": "Top 10% income composition-Dividends",
    "Interest": "Top 10% income composition-Interest Income",
    "Rent": "Top 10% income composition-Rents",
    "Business": "Top 10% income composition-Entrepreneurial income",
}
years_c, stacks = None, []
# align on common years
maps = {k: dict(timeseries(us_rows, col)) for k, col in COMP_COLS.items()}
common = sorted(set.intersection(*[set(m) for m in maps.values()]))
fig, ax = plt.subplots()
ys = [np.array([maps[k][y] for y in common], dtype=float) for k in COMP_COLS]
ax.stackplot(common, *ys, labels=list(COMP_COLS),
             colors=["#1f4e79", "#2e7d32", "#c62828", "#00838f", "#ef6c00"], alpha=0.9)
ax.set_ylim(0, 100)
ax.set_ylabel("Percentage of top-10% income")
ax.set_title("U.S. Top 10% Income Composition")
ax.legend(loc="upper left", frameon=False, ncol=5, fontsize=8)
fig.tight_layout()
fig.savefig("topinc_us_composition.png", dpi=140)
plt.show()


**Reading.** Wages dominate the top 10% by the late 20th century. Dividends / interest / rent shrink as a share after mid-century; entrepreneurial income dips then recovers after ~1980 (Cookbook reading: possible tech / pass-through boom). Composition of the *top 0.1%* is much more capital-heavy — a useful extra practice plot.


## 6. Alternates

In [ ]:
df = pd.read_csv(DATA)
us = df[df["Country"] == "United States"].sort_values("Year")
print(us.shape, us["Year"].min(), us["Year"].max())
print("Top 1% 1980/2007/2008",
      us.loc[us.Year == 1980, "Top 1% income share"].iloc[0],
      us.loc[us.Year == 2007, "Top 1% income share"].iloc[0],
      us.loc[us.Year == 2008, "Top 1% income share"].iloc[0])

ax = us.set_index("Year")[list(SHARE_COLS)].plot(lw=2)
ax.set_ylabel("Percentage of total income")
ax.set_title("Alternate: pandas.DataFrame.plot — U.S. top shares")
plt.tight_layout(); plt.show()


**NumPy gotcha (Cookbook §import).** `np.recfromcsv` / `genfromtxt` are not CSV-quote aware. Headers such as `"Top 10% income composition-Wages, salaries and pensions"` split on the comma, Country becomes `nan`, Year becomes float. Prefer `csv` or pandas; if you must use NumPy, pass an explicit `names=` / `dtype=` and `skip_header=1`.

**R-shaped alternate** (Cookbook last recipe):

```r
id2 <- read.csv("data/income_dist.csv", header=TRUE, check.names=FALSE)
id_us <- id2[id2$Country == "United States", ]
shares <- ts(id_us[, c("Top 10% income share","Top 5% income share",
                       "Top 1% income share","Top 0.5% income share",
                       "Top 0.1% income share")], start=1913, frequency=1)
plot.ts(shares, plot.type="single")
```

`check.names=FALSE` is mandatory — otherwise R turns `%` and spaces into dots.


## 7. More practice

In [ ]:
FOCUS = ["Sweden", "Norway", "Ireland", "Italy", "Japan", "Spain", "France", "United States"]

def nearest(frame, year, col, window=2):
    sl = frame.loc[frame.Year.between(year - window, year + window), ["Year", col]].dropna()
    return None if sl.empty else sl.iloc[(sl.Year - year).abs().argmin()]

rows = []
for c in FOCUS:
    sub = df[df.Country == c]
    a = nearest(sub, 1980, "Top 1% income share")
    late = sub.loc[sub.Year.between(1999, 2010), ["Year", "Top 1% income share"]].dropna()
    if a is None or late.empty:
        print("skip", c); continue
    y1, v1 = int(late.iloc[-1].Year), float(late.iloc[-1]["Top 1% income share"])
    rows.append({"country": c, "y0": int(a.Year), "v0": float(a["Top 1% income share"]),
                 "y1": y1, "v1": v1, "dpp": v1 - float(a["Top 1% income share"])})
cmp = pd.DataFrame(rows).sort_values("dpp")
print(cmp.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(cmp))
ax.hlines(y, cmp.v0, cmp.v1, color="#90a4ae", lw=3)
ax.scatter(cmp.v0, y, color="#0277bd", s=50, label="~1980", zorder=3)
ax.scatter(cmp.v1, y, color="#c62828", s=50, label="latest ≤2010", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(cmp.country)
ax.set_xlabel("Top 1% income share (%)")
ax.set_title("Top 1% share: circa 1980 vs latest (WTID total income)")
ax.legend(frameon=False)
for i, r in enumerate(cmp.itertuples()):
    ax.text(max(r.v0, r.v1) + 0.15, i, f"{r.dpp:+.1f} pp ({r.y1})", va="center", fontsize=8)
fig.tight_layout(); fig.savefig("topinc_cross_country.png", dpi=140); plt.show()


**Reading vs Dialnet (2014).** The Spanish *Encrucijadas* graphic used *wage* income only and reported U.S. +9.4 pp (8.0 → 17.4-ish) versus Spain +0.7 pp. Our WTID *total-income* Top 1% move is **U.S. +9.5 pp (8.18 in 1980 → 17.67 in 2008)** and **Spain +0.9 pp (7.75 in 1982 → 8.61 in 2008)**. Direction and ranking match; levels differ because capital income is in our numerator.


In [ ]:
sp_years = [1981, 1986, 1991, 1996, 2001, 2006, 2008]
sp = df[(df.Country == "Spain") & (df.Year.isin(sp_years))][
    ["Year", "Top 1% income share", "Top 0.1% income share", "Average income per tax unit"]
].copy()
sp["ratio_0_1_to_avg"] = (
    # average of top 0.1% ≈ (share/100) * mean / 0.001
    (sp["Top 0.1% income share"] / 100.0) * sp["Average income per tax unit"] / 0.001
    / sp["Average income per tax unit"]
)
print(sp.to_string(index=False))
print("implied top 0.1% / overall mean = share / 0.1")


In [ ]:
CANDIDATES = [
    "Top 1% income share",
    "Top 1% income share-tax units",
    "Top 1% income share-adults",
    "Top 1% income share-LAD",
]
def best_top1_col(frame, country):
    sub = frame[frame.Country == country]
    scored = [(col, sub[col].notna().sum()) for col in CANDIDATES if col in frame.columns]
    scored.sort(key=lambda kv: kv[1], reverse=True)
    return scored[0] if scored else None

for c in ["United Kingdom", "United States", "Argentina", "Spain"]:
    print(c, best_top1_col(df, c))


## 8. Simulation

In [ ]:
# --- knobs ---
NOISE_SD = 0.30
START, END = 1980, 2007
N_DRAWS = 2000
COUNTRY = "United States"
COL = "Top 1% income share"
# ---------------
rng = np.random.default_rng(42)
obs = df.loc[df.Country == COUNTRY, ["Year", COL]].dropna().sort_values("Year")
v0 = float(obs.loc[obs.Year == START, COL].iloc[0])
v1 = float(obs.loc[obs.Year == END, COL].iloc[0])
point = v1 - v0

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(obs.Year, obs[COL], color="#1f4e79", lw=2, label="observed")
for _ in range(30):
    axes[0].plot(obs.Year, obs[COL].to_numpy() + rng.normal(0, NOISE_SD, len(obs)),
                 color="#90caf9", alpha=0.25, lw=0.8)
axes[0].set_title(f"{COUNTRY} {COL}\nnoise sd={NOISE_SD} pp")
axes[0].set_ylabel("Share (%)")
axes[0].legend(frameon=False)

draws = (v1 + rng.normal(0, NOISE_SD, N_DRAWS)) - (v0 + rng.normal(0, NOISE_SD, N_DRAWS))
axes[1].hist(draws, bins=30, color="#1f4e79", alpha=0.85)
axes[1].axvline(point, color="#c62828", lw=2, label=f"point {point:.2f} pp")
axes[1].set_title(f"Change {START}→{END} under endpoint noise")
axes[1].set_xlabel("pp change")
axes[1].legend(frameon=False)
fig.tight_layout(); fig.savefig("topinc_simulation.png", dpi=140); plt.show()
print(f"point {point:.2f} pp | sim mean {draws.mean():.2f} | 5-95 {np.quantile(draws,[0.05,0.95])}")


**How to use the knobs.** Raise `NOISE_SD` to 1.0 and the 1980–2007 U.S. jump is still far from zero — the finding is not an artifact of a 0.3 pp rounding error. Swap `COUNTRY` to `"Spain"` (use 1982 and 2008) and the same noise *can* swallow the +0.9 pp move. That is the policy-relevant contrast.


## 9. Mini report layer

In [ ]:
try:
    from jinja2 import Template
    tmpl = Template(
        "<html><body><h1>{{ title }}</h1>"
        "<p>U.S. Top 1% share: {{ v0 }}% ({{ y0 }}) → {{ v1 }}% ({{ y1 }}), "
        "change {{ d }} pp.</p></body></html>"
    )
    html = tmpl.render(title="WTID snapshot", y0=1980, v0=8.18, y1=2007, v1=18.29, d=10.11)
except Exception:
    html = (
        "<html><body><h1>WTID snapshot</h1>"
        "<p>U.S. Top 1% share: 8.18% (1980) → 18.29% (2007), change 10.11 pp.</p>"
        "</body></html>"
    )
Path("topinc_mini_report.html").write_text(html)
print("wrote topinc_mini_report.html")


## 10. Audience rewrite

In [ ]:
expert = (
    "Using WTID fiscal-income series with fractiles defined on income excluding capital gains, "
    "the U.S. top 1% share rose from 8.18% in 1980 to 18.29% in 2007 (17.67% in 2008). "
    "The path is the familiar U-shape: pre-war highs, postwar compression, post-1980 recovery of the upper tail. "
    "Capital-gains-inclusive series lie above these levels and are more cyclical; Pareto-Lorenz b also rose, "
    "consistent with a thicker tail rather than a uniform shift of the top decile."
)
technician = (
    "Refresh path: `data/income_dist.csv` → filter Country=='United States' → "
    "column 'Top 1% income share' (not the -adults / -tax units variants). "
    "Cast blanks to NA before any delta. Re-rank warning: do not subtract including-CG from excluding-CG "
    "and call it 'the CG of the same people'. Unit of the average-income columns is 2008 USD; "
    "notes.csv documents the income control. The generator helpers in TopInc_Py.py are the refresh surface."
)
executive = (
    "In the United States the top 1% of tax units went from about eight cents of every income dollar in 1980 "
    "to about eighteen cents in 2007. That is a ten-point swing in one generation, several times larger than "
    "the move in Spain or France on the same database. The 2008 crisis clipped the last year but did not "
    "return the series to the 1970s plateau. Average incomes at the very top pulled away from the rest of the distribution after 1980."
)
nonspecialist = (
    "Tax records let researchers add up how much of all income goes to the highest-earning hundredth of households. "
    "In the U.S. that slice held a bit more than 8% of income in 1980 and about 18% in 2007. "
    "Put differently: the gap between the typical household and the highest-earning hundredth widened a lot "
    "over those 27 years. Other rich countries in the same file moved in the same direction, but much less."
)
for label, text in [("EXPERT", expert), ("TECHNICIAN", technician),
                    ("EXECUTIVE", executive), ("NONSPECIALIST", nonspecialist)]:
    print("====", label, "====")
    print(text)
    print()


## Key numerical anchors (this extract)

| Series | 1913/17 | 1980 | 2007 | 2008 |
|--------|---------|------|------|------|
| US Top 10% share | 40.29 (1917) | 32.87 | 45.51 | 45.60 |
| US Top 1% share | 17.96 | 8.18 | 18.29 | 17.67 |
| US Top 0.1% share | 8.62 | 2.23 | 8.23 | 7.77 |
| US avg income / tax unit (2008 USD) | 14,602 | 40,748 | 54,080 | 51,255 |
| Spain Top 1% share | — | 7.75 (1982) | — | 8.61 |
